In [2]:
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

# Load the image
image_path = "../debug_images/iter_3500/render_image.png"
image = Image.open(image_path).convert("RGB")

# Transform the image to a tensor
transform = transforms.ToTensor()
image_tensor = transform(image)

In [16]:
midas_dir = '../pretrained_models/midas'
# midas_dir = './pretrained_models/MiDaS'

midas = torch.hub.load(midas_dir, "DPT_Hybrid", source='local')
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
midas.to(device)
midas.eval()
for param in midas.parameters():
    param.requires_grad = False

midas_transforms = torch.hub.load(midas_dir, "transforms", source='local')
transform = midas_transforms.dpt_transform
downsampling = 1

def estimate_depth(img, mode='test'):
    h, w = img.shape[1:3]  #h:378,w:504 for fern image_8
    norm_img = (img[None] - 0.5) / 0.5
    norm_img = torch.nn.functional.interpolate(
        norm_img,
        size=(384, 512),
        mode="bicubic",
        align_corners=False)

    if mode == 'test':
        with torch.no_grad():
            prediction = midas(norm_img)
            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=(h//downsampling, w//downsampling),
                mode="bicubic",
                align_corners=False,
            ).squeeze()
    else:
        prediction = midas(norm_img)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=(h//downsampling, w//downsampling),
            mode="bicubic",
            align_corners=False,
        ).squeeze()
    return prediction

image_tensor = image_tensor.to(device)
image_tensor.requires_grad = True

# Estimate depth
for i in range(1000):
    print("current iter: ", i)
    depth = estimate_depth(image_tensor, mode='train')
    
    # Create a tensor of ones with the same shape as depth
    target = torch.ones_like(depth, device=device)
    
    # Calculate mean squared error
    mse_loss = torch.nn.functional.mse_loss(depth, target)
    
    # Backpropagate the loss
    mse_loss.backward()
    
tensor = depth.cpu().detach()
tensor_normalized = (tensor - tensor.min()) / (tensor.max() - tensor.min())
transform = transforms.ToPILImage()
image = transform(tensor_normalized)
image.save("./test_depth.png")
# # Convert depth to numpy array
# depth = depth.cpu().numpy()
# # Normalize depth for visualization
# depth_normalized = (depth - depth.min()) / (depth.max() - depth.min())
# # Convert to uint8
# depth_normalized = (depth_normalized * 255).astype('uint8') 
# # Convert to PIL Image
# depth_image = Image.fromarray(depth_normalized)
# #show the depth image
# plt.imshow(depth_image, cmap='gray')

/home/jiahao/anaconda3/envs/ipsm/lib/python3.10/site-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name vit_base_resnet50_384 to current vit_base_r50_s16_384.orig_in21k_ft_in1k.
  model = create_fn(


current iter:  0
current iter:  1
current iter:  2
current iter:  3
current iter:  4
current iter:  5
current iter:  6
current iter:  7
current iter:  8
current iter:  9
current iter:  10
current iter:  11
current iter:  12
current iter:  13
current iter:  14
current iter:  15
current iter:  16
current iter:  17
current iter:  18
current iter:  19
current iter:  20
current iter:  21
current iter:  22
current iter:  23
current iter:  24
current iter:  25
current iter:  26
current iter:  27
current iter:  28
current iter:  29
current iter:  30
current iter:  31
current iter:  32
current iter:  33
current iter:  34
current iter:  35
current iter:  36
current iter:  37
current iter:  38
current iter:  39
current iter:  40
current iter:  41
current iter:  42
current iter:  43
current iter:  44
current iter:  45
current iter:  46
current iter:  47
current iter:  48
current iter:  49
current iter:  50
current iter:  51
current iter:  52
current iter:  53
current iter:  54
current iter:  55
cu

KeyboardInterrupt: 